# Pipeline — DML na Camada Bronze (Delta Lake)

Operações de INSERT incremental, UPDATE, DELETE e MERGE (UPSERT) sobre as tabelas Delta Lake do bucket `bronze`.

Domínio: **BikeStores** — lojas, produtos, pedidos e estoques.

## 1. Configuração

In [1]:
MINIO_ENDPOINT   = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BRONZE_BUCKET    = "s3a://bronze"

## 2. SparkSession com suporte a S3A e Delta Lake

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("bronze_dml")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession iniciada — versão: {spark.version}")

your 131072x1 screen size is bogus. expect trouble
26/05/06 06:56:52 WARN Utils: Your hostname, Ismael resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 06:56:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ismasel/projects/eng-de-dados/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ismasel/.ivy2/cache
The jars for the packages stored in: /home/ismasel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-197d066a-29ab-46b6-872b-44ac43b3062e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 205ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr

SparkSession iniciada — versão: 3.5.0


## 3. Carregamento das DeltaTables

In [3]:
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import (
    col, lit, current_timestamp, current_date,
    datediff, round as spark_round,
)
from pyspark.sql.types import IntegerType, DoubleType

delta_products = DeltaTable.forPath(spark, f"{BRONZE_BUCKET}/products")
delta_orders   = DeltaTable.forPath(spark, f"{BRONZE_BUCKET}/orders")
delta_stocks   = DeltaTable.forPath(spark, f"{BRONZE_BUCKET}/stocks")

print("DeltaTables carregadas.")

26/05/06 06:57:04 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DeltaTables carregadas.


## 4. INSERT — Novos produtos (catálogo 2019)

Simula a chegada de produtos ainda não presentes no bronze.

In [4]:
new_products = [
    Row(product_id=322, product_name="Trek Checkpoint ALR 4 - 2019",
        brand_id=9, category_id=7, model_year=2019, list_price=1499.99,
        source_system="catalog_2019"),
    Row(product_id=323, product_name="Surly Midnight Special - 2019",
        brand_id=8, category_id=7, model_year=2019, list_price=1699.00,
        source_system="catalog_2019"),
]

df_new_products = (
    spark.createDataFrame(new_products)
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("extracted_at",       current_timestamp())
    .withColumn("product_id",  col("product_id").cast(IntegerType()))
    .withColumn("brand_id",    col("brand_id").cast(IntegerType()))
    .withColumn("category_id", col("category_id").cast(IntegerType()))
    .withColumn("model_year",  col("model_year").cast(IntegerType()))
    .withColumn("list_price",  col("list_price").cast(DoubleType()))
)

(
    df_new_products.write
    .format("delta")
    .mode("append")
    .partitionBy("model_year")
    .save(f"{BRONZE_BUCKET}/products")
)

total = spark.read.format("delta").load(f"{BRONZE_BUCKET}/products").count()
print(f"INSERT concluído. Total de produtos: {total}")

26/05/06 06:57:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


INSERT concluído. Total de produtos: 323


## 5. UPDATE — Ajuste de preços via DeltaTable API

In [5]:
# Desconto de 15% nos produtos do model_year 2016 (liquidação de estoque antigo)
delta_products.update(
    condition=col("model_year") == 2016,
    set={
        "list_price":         spark_round(col("list_price") * 0.85, 2),
        "bronze_ingested_at": current_timestamp(),
    },
)

print("UPDATE executado — preços do model_year 2016 reduzidos em 15%.")
(
    delta_products.toDF()
    .filter(col("model_year") == 2016)
    .select("product_id", "product_name", "list_price")
    .limit(5)
    .show(truncate=False)
)

UPDATE executado — preços do model_year 2016 reduzidos em 15%.
+----------+----------------------------------+----------+
|product_id|product_name                      |list_price|
+----------+----------------------------------+----------+
|1         |Trek 820 - 2016                   |322.99    |
|2         |Ritchey Timberwolf Frameset - 2016|637.49    |
|3         |Surly Wednesday Frameset - 2016   |849.99    |
|4         |Trek Fuel EX 8 29 - 2016          |2464.99   |
|5         |Heller Shagamaw Frame - 2016      |1122.84   |
+----------+----------------------------------+----------+



In [6]:
# Pedidos pendentes (status 1) vencidos e sem envio → Rejected (status 3)
delta_orders.update(
    condition=(
        (col("order_status") == 1)
        & (col("required_date").cast("date") < current_date())
        & col("shipped_date").isNull()
    ),
    set={
        "order_status":      lit(3),
        "bronze_ingested_at": current_timestamp(),
    },
)

print("UPDATE executado — pedidos pendentes vencidos marcados como Rejected.")
(
    delta_orders.toDF()
    .filter(col("order_status") == 3)
    .select("order_id", "order_status", "order_date", "required_date")
    .limit(5)
    .show()
)

UPDATE executado — pedidos pendentes vencidos marcados como Rejected.
+--------+------------+----------+-------------+
|order_id|order_status|order_date|required_date|
+--------+------------+----------+-------------+
|    1512|           3|2018-04-09|   2018-04-09|
|    1515|           3|2018-04-10|   2018-04-10|
|    1481|           3|2018-04-01|   2018-04-01|
|    1482|           3|2018-04-01|   2018-04-01|
|    1483|           3|2018-04-02|   2018-04-02|
+--------+------------+----------+-------------+



## 6. DELETE — Remoção seletiva de registros

In [7]:
antes = delta_products.toDF().count()
print(f"Produtos antes do DELETE: {antes}")

# Remove produtos anteriores a 2016
delta_products.delete(condition=col("model_year") < 2016)

depois = delta_products.toDF().count()
print(f"Produtos após o DELETE:   {depois}")
print(f"{antes - depois} produto(s) removido(s).")

Produtos antes do DELETE: 323
Produtos após o DELETE:   323
0 produto(s) removido(s).


In [8]:
# Remove pedidos rejeitados com mais de 2 anos
delta_orders.delete(
    condition=(
        (col("order_status") == 3)
        & (datediff(current_date(), col("order_date").cast("date")) > 730)
    )
)

print("DELETE executado — pedidos rejeitados com mais de 2 anos removidos.")

DELETE executado — pedidos rejeitados com mais de 2 anos removidos.


## 7. MERGE (UPSERT) — Sincronização incremental de estoques

Atualiza a quantidade se o par `(store_id, product_id)` já existe; insere caso contrário.

In [9]:
stock_updates = [
    Row(store_id=1, product_id=1,   quantity=30, source_system="wms"),
    Row(store_id=2, product_id=5,   quantity=12, source_system="wms"),
    Row(store_id=1, product_id=322, quantity=5,  source_system="wms"),
    Row(store_id=2, product_id=323, quantity=3,  source_system="wms"),
]

df_stock_updates = (
    spark.createDataFrame(stock_updates)
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("extracted_at",       current_timestamp())
    .withColumn("store_id",   col("store_id").cast(IntegerType()))
    .withColumn("product_id", col("product_id").cast(IntegerType()))
    .withColumn("quantity",   col("quantity").cast(IntegerType()))
)

(
    delta_stocks.alias("target")
    .merge(
        df_stock_updates.alias("source"),
        (col("target.store_id")   == col("source.store_id"))
        & (col("target.product_id") == col("source.product_id")),
    )
    .whenMatchedUpdate(set={
        "quantity":           col("source.quantity"),
        "bronze_ingested_at": current_timestamp(),
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"MERGE concluído. Total em stocks: {delta_stocks.toDF().count()}")

MERGE concluído. Total em stocks: 941


## 8. Time Travel — Histórico de versões

In [10]:
delta_products.history().select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                             |
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
# Snapshot da versão 0 — carga inicial vinda do landing-zone
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(f"{BRONZE_BUCKET}/products")
)
print(f"Versão 0 (carga inicial): {df_v0.count()} produtos")
df_v0.select("product_id", "product_name", "model_year", "list_price").show(5, truncate=False)

Versão 0 (carga inicial): 321 produtos
+----------+----------------------------------+----------+----------+
|product_id|product_name                      |model_year|list_price|
+----------+----------------------------------+----------+----------+
|1         |Trek 820 - 2016                   |2016      |379.99    |
|2         |Ritchey Timberwolf Frameset - 2016|2016      |749.99    |
|3         |Surly Wednesday Frameset - 2016   |2016      |999.99    |
|4         |Trek Fuel EX 8 29 - 2016          |2016      |2899.99   |
|5         |Heller Shagamaw Frame - 2016      |2016      |1320.99   |
+----------+----------------------------------+----------+----------+
only showing top 5 rows



In [12]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
